# Assignment 2 - Numbers to Neurons

---
> Complete each question below. Write theory answers in Markdown cells and code in code cells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

---
## Q1 - Perceptrons and Sigmoid Neurons

### Q1a - NAND Perceptron

Write a Python class using NumPy that implements a **NAND logic gate** using a single perceptron. Your implementation must use **vectorized operations** (no explicit `for` loops in the activation function) to process an input matrix of shape `(N, 2)` and return the corresponding binary outputs.

Mathematically determine and set the correct weights and bias within your code.

In [ ]:
import numpy as np

class NANDPerceptron:
    def __init__(self):

        self.weights = np.array([-1.0, -1.0])
        self.bias = 1.5

    def predict(self, X):

        X = np.asarray(X)

        net_input = X @ self.weights + self.bias

        return (net_input >= 0).astype(int)


### Q1b - Perceptron Sensitivity

Explain precisely why a tiny change in one weight can cause **chaos** in a perceptron network. What property of the perceptron makes this happen?

> **Your answer:**A perceptron makes decisions using a hard threshold: it outputs either 0 or 1, with no values in between.

Because of this, if the weighted sum of inputs is very close to the threshold, even a tiny change in one weight can push it from one side of the threshold to the other. The output then suddenly flips from 0 to 1 or from 1 to 0.

In a network, that flipped output becomes an input to other perceptrons, causing more outputs to change. This chain reaction can make a very small weight change produce a very large change in the final result.

### Q1c - Sigmoid to Perceptron Limit

Suppose you multiply all the weights and bias of a sigmoid neuron by a positive constant c > 0.

1. As c approaches what value will this sigmoid neuron behave exactly like a classic perceptron? Why mathematically?
2. Explain in 2-3 lines why we prefer sigmoid neurons over perceptrons.

> **Your answer:**If all the weights and the bias of a sigmoid neuron are multiplied by a positive constant that becomes extremely large, the sigmoid curve becomes steeper and steeper. In the limit, it turns into an almost perfect on/off switch: outputs become very close to 1 on one side of the threshold and very close to 0 on the other side. At that point, the sigmoid neuron behaves like a classic perceptron.

We prefer sigmoid neurons because their output changes smoothly when the weights change. This smoothness allows learning algorithms to determine how to adjust the weights effectively. A perceptron changes its output abruptly, which makes it unsuitable for gradient-based training methods such as backpropagation.

---
## Q2 - The Quadratic Cost Function

$$C(w,b) = \frac{1}{2n} \sum_x \|y(x) - a\|^2$$

### Q2a - The Factor of 1/2

Why is there a factor of 1/2 in front? Derive why it is mathematically convenient when computing the derivative of C with respect to any weight w.

> **Your answer:**You can copy the following directly:

The factor of `1/2` is included for mathematical convenience when differentiating the cost function.

Suppose the cost function is


C = (1/2n) Σ (y(x) - a)^2


where the summation is over all training examples.

When we differentiate `C` with respect to a weight `w`, we need to differentiate the squared term:

d/dw [(y - a)^2]
= 2(y - a) · d(y - a)/dw


The derivative of the square produces a factor of `2`.

Substituting into the derivative of the cost function:


dC/dw
= (1/2n) Σ [2(y - a) · d(y - a)/dw]


The `2` from differentiation cancels the `1/2` already present in the cost function:


dC/dw
= (1/n) Σ [(y - a) · d(y - a)/dw]


Without the factor `1/2`, we would obtain


dC/dw
= (2/n) Σ [(y - a) · d(y - a)/dw]


which contains an unnecessary factor of `2` in every gradient expression.

Therefore, the factor `1/2` does not affect the location of the minimum of the cost function. It is included simply to make the derivatives cleaner by canceling the factor of `2` that arises when differentiating a squared term.



### Q2b - Computing C by Hand

A network has two training examples:
- Example 1: output = 0.9, target = 1.0
- Example 2: output = 0.3, target = 0.0

Compute C by hand and explain what the value tells you about network performance.

> **Your answer:**Using the cost function
C = (1/2n) Σ (y - a)^2

where:

* `n = 2` training examples
* Example 1: `y = 0.9`, `a = 1.0`
* Example 2: `y = 0.3`, `a = 0.0`

Compute the squared errors:

(0.9 - 1.0)^2 = (-0.1)^2 = 0.01

(0.3 - 0.0)^2 = (0.3)^2 = 0.09


Sum them:

0.01 + 0.09 = 0.10

Now apply the factor `1/(2n)`:

C = (1/(2×2)) × 0.10
C = (1/4) × 0.10



C = 0.025

In [ ]:
import numpy as np

# Verify your hand computation
outputs = np.array([0.9, 0.3])
targets = np.array([1.0, 0.0])

n = len(outputs)
C_val = (1 / (2 * n)) * np.sum((outputs - targets) ** 2)

print(f'C = {C_val}')

### Q2c - Cost vs Accuracy

Explain why we use a cost function and do **not** directly maximise classification accuracy.

> **Your answer:**We use a cost function instead of accuracy because accuracy is not smooth-small changes in the weights often cause no change in accuracy. A cost function changes continuously with the network's output, providing useful gradients that allow learning algorithms like gradient descent and backpropagation to update the weights effectively.


### Q2d - Absolute Error vs Squared Error

An alternative cost is the absolute error: C = (1/n) * sum |y(x) - a|.

Explain **one mathematical problem** with using absolute error instead of squared error, specifically when doing gradient descent.

> **Your answer:**A mathematical problem with absolute error is that it is not differentiable at zero (when the prediction exactly equals the target).

Since gradient descent relies on computing derivatives to determine how to update the weights, the derivative of the absolute error is undefined at that point. In contrast, the squared error is smooth and differentiable everywhere, making optimization easier and more stable.


---
## Q3 - Gradient Descent on a Toy Cost Function

$$C(w, b) = (w - 3)^2 + 2(b - 1)^2$$

### Q3a - Minimum by Inspection

By inspection, what values of w and b minimise C? What is the minimum value of C? Explain your reasoning without calculus.

> **Your answer:**The cost function is (C(w,b) = (w-3)^2 + 2(b-1)^2). Since both terms are squares, they can never be negative, and the smallest value each square can take is 0. The first term becomes 0 when (w=3), and the second term becomes 0 when (b=1). Therefore, the cost is minimized when (w=3) and (b=1), giving (C(3,1)=0). Hence, the minimum value of the cost function is 0, achieved at (w=3) and (b=1).



### Q3b - Two Steps by Hand

Compute dC/dw and dC/db. Starting from (w, b) = (0, 0) with learning rate eta = 0.1, show the first two gradient descent update steps by hand. Where does the parameter vector end up after step 2?

> **Your answer:**The cost function is (C(w,b)=(w-3)^2+2(b-1)^2). Differentiating, we obtain (\frac{dC}{dw}=2(w-3)) and (\frac{dC}{db}=4(b-1)). Starting from ((w,b)=(0,0)) with learning rate (\eta=0.1), the gradients are (\frac{dC}{dw}=-6) and (\frac{dC}{db}=-4). Updating the parameters gives (w=0-0.1(-6)=0.6) and (b=0-0.1(-4)=0.4). For the second step, at ((w,b)=(0.6,0.4)), the gradients are (\frac{dC}{dw}=2(0.6-3)=-4.8) and (\frac{dC}{db}=4(0.4-1)=-2.4). Updating again gives (w=0.6-0.1(-4.8)=1.08) and (b=0.4-0.1(-2.4)=0.64). Therefore, after two gradient descent steps, the parameter vector is ((1.08,,0.64)).


In [ ]:
def cost(w, b):
    return (w - 3)**2 + 2*(b - 1)**2

def grad_cost(w, b):
    dw = 2*(w - 3)
    db = 4*(b - 1)
    return dw, db

# Verify
eta = 0.1
w, b = 0.0, 0.0
for step in range(1, 3):
    dw, db = grad_cost(w, b)
    w = w - eta * dw
    b = b - eta * db
    print(f'Step {step}: w={w:.4f}, b={b:.4f}, C={cost(w,b):.4f}')

### Q3c - Too-Large Learning Rate

If you set eta = 0.6, what goes wrong on the b update? Show numerically. What does this tell you about learning rate selection?

> **Your answer:**For the (b)-parameter, the gradient is (\frac{dC}{db}=4(b-1)). Starting from (b=0) with learning rate (\eta=0.6), the gradient is (-4), so the update gives (b=0-0.6(-4)=2.4). Notice that the true minimum is at (b=1), but the update jumps past it to (b=2.4). On the next step, the gradient becomes (4(2.4-1)=5.6), so (b=2.4-0.6(5.6)=-0.96), which jumps past the minimum again to the other side. Thus, the values oscillate around the optimum instead of smoothly approaching it. This shows that if the learning rate is too large, gradient descent can overshoot the minimum, causing instability or even divergence.


In [ ]:
eta_bad = 0.6
w, b = 0.0, 0.0
for step in range(1, 5):
    dw, db = grad_cost(w, b)
    w = w - eta_bad * dw
    b = b - eta_bad * db
    print(f'Step {step}: w={w:.4f}, b={b:.4f}, C={cost(w,b):.4f}')

---
## Q4 - Sigmoid Neuron: Implementation and Visualisation

In [ ]:
class SigmoidNeuron:

    def __init__(self, weights, bias):
        self.weights = np.array(weights, dtype=float)
        self.bias = float(bias)

    def forward(self, x):
        # TODO: compute z = w.x + b, return sigmoid(z)
        pass

    def numerical_gradient(self, x, h=1e-5):
        # TODO: centered difference (f(x+h) - f(x-h)) / 2h
        # return (grad_weights, grad_bias)
        pass

### Q4a - Forward Pass and Gradients

For weights=[2, -1] and bias=0.5, compute the output and all gradients at x=[1, 1]. Explain what the gradient w.r.t. w1 means in plain English.

In [ ]:
neuron = SigmoidNeuron(weights=[2, -1], bias=0.5)
x = np.array([1.0, 1.0])

output = neuron.forward(x)
grad_w, grad_b = neuron.numerical_gradient(x)

print(f'Output:        {output}')
print(f'd(output)/dw1: {grad_w[0]:.6f}')
print(f'd(output)/dw2: {grad_w[1]:.6f}')
print(f'd(output)/db:  {grad_b:.6f}')

**Explanation of gradient w.r.t. w1:**

> **Your answer:**

For weights = [2, -1] and bias = 0.5, the neuron computes (z = 2(1) + (-1)(1) + 0.5 = 1.5). Applying the sigmoid function gives an output of approximately 0.817574. The gradients are d(output)/dw1 = 0.149146,d(output)/dw2 = 0.149146 and d(output)/db  = 0.149146. The gradient with respect to (w_1) measures how sensitive the neuron's output is to changes in the first weight. A value of 0.149146 means that a small increase in (w_1) will increase the output by approximately 0.149146 times the size of that change (for very small changes), while a small decrease in (w_1) will decrease the output by approximately the same amount.


### Q4b - Plot sigma(z) and sigma'(z)

Plot sigma(z) and its derivative sigma'(z) = sigma(z)(1 - sigma(z)) for z in [-10, 10].

- At what value of z is the gradient largest?
- What happens as z approaches +/- infinity and why is this a problem (vanishing gradient)?

> **Your answer:**The derivative of the sigmoid function is σ
′
(z)=σ(z)(1−σ(z)). The gradient is largest at z=0, where σ(0)=0.5, giving a maximum gradient of σ
′
(0)=0.25. As z→+∞, the sigmoid output approaches 1 and the gradient approaches 0. Similarly, as z→−∞, the sigmoid output approaches 0 and the gradient again approaches 0. This is problematic because during backpropagation the gradients become extremely small, causing the weights to update very slowly or not at all. This phenomenon is known as the vanishing gradient problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-10, 10, 500)

sigma = 1 / (1 + np.exp(-z))
sigma_prime = sigma * (1 - sigma)

plt.figure(figsize=(8, 4))
plt.plot(z, sigma, label='sigma(z)')
plt.plot(z, sigma_prime, label="sigma'(z)")
plt.xlabel('z')
plt.legend()
plt.title('Sigmoid and its derivative')
plt.grid(True)
plt.show()

### Q4c - Is sigma(100z) basically a perceptron?

A classmate claims: 'a sigmoid neuron with weights multiplied by 100 is basically the same as a perceptron.'

Plot sigma(100z) vs sigma(z) vs the step function. Is the claim correct? Under what condition does it break down?

> **Your answer:**The claim is mostly correct. Multiplying the weights by 100 makes the sigmoid curve extremely steep around z=0, so σ(100z) closely resembles a perceptron's step function. For most values of z, the output is very close to either 0 or 1.

However, the claim breaks down near the decision boundary z=0. A perceptron changes output abruptly at the threshold, while σ(100z) remains continuous and differentiable. Also, because σ(100z) saturates quickly, its gradient becomes nearly zero for most values of z, leading to the vanishing gradient problem. Thus, a steep sigmoid can behave similarly to a perceptron in terms of outputs, but it is not mathematically identical and still differs near the threshold and during training.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-2, 2, 1000)

# Compute sigma(z), sigma(100z), and step function
sigma = 1 / (1 + np.exp(-z))
sigma_100 = 1 / (1 + np.exp(-100 * z))
step = (z >= 0).astype(int)

plt.figure(figsize=(8, 4))
plt.plot(z, sigma, label='sigma(z)')
plt.plot(z, sigma_100, label='sigma(100z)')
plt.plot(z, step, label='Step Function')
plt.title('sigma(z) vs sigma(100z) vs Step Function')
plt.xlabel('z')
plt.legend()
plt.grid(True)
plt.show()

---
## Q5 - Gradient Descent on a Toy Regression Problem

A single-layer network (one input, one output, no activation) must learn y = 3x + 1.

In [ ]:
X_train = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y_train = np.array([1.0, 4.0, 7.0, 10.0, 13.0])

### Q5a - Analytical Gradients + Full-Batch GD

Derive dC/dw and dC/db analytically (show algebra). Implement full-batch GD with eta = 0.01 for 200 epochs. Plot the loss curve. Do w and b converge to 3 and 1?

**Derivation (show algebra):**

> **Your answer:**For the linear model ŷ = wx + b and the cost function C(w,b) = (1/2n) Σ(wxᵢ + b − yᵢ)², the analytical gradients are:

∂C/∂w = (1/n) Σ[(wxᵢ + b − yᵢ)xᵢ]

∂C/∂b = (1/n) Σ(wxᵢ + b − yᵢ)

Using full-batch gradient descent with η = 0.01 for 200 epochs, the parameters converge close to w = 3 and b = 1. This is expected because the training data exactly follows the relationship y = 3x + 1. As training progresses, the loss decreases steadily toward zero, indicating that the model is learning the correct slope and intercept and successfully fitting the data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_train = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y_train = np.array([1.0, 4.0, 7.0, 10.0, 13.0])

w = 0.0
b = 0.0

eta = 0.01
epochs = 200

losses = []
n = len(x_train)

for epoch in range(epochs):

    y_pred = w * x_train + b
    error = y_pred - y_train

    loss = (1/(2*n)) * np.sum(error**2)
    losses.append(loss)

    dC_dw = (1/n) * np.sum(error * x_train)
    dC_db = (1/n) * np.sum(error)

    w = w - eta * dC_dw
    b = b - eta * dC_db

print(f"Final w = {w:.6f}")
print(f"Final b = {b:.6f}")

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Gradient Descent Loss Curve")
plt.grid(True)
plt.show()

### Q5b - SGD with Mini-Batch Size = 1

Implement SGD with batch size 1. Train 200 epochs with eta = 0.01. Plot the loss curve. How does it differ from full-batch? Explain in terms of gradient quality and noise.

> **Your answer:**

In [ ]:
w, b = 0.0, 0.0
eta = 0.01
losses_sgd = []

n = len(x_train)

for epoch in range(200):

    idx = np.random.randint(n)
    x = x_train[idx]
    y = y_train[idx]


    y_pred = w * x + b
    error = y_pred - y


    dC_dw = error * x
    dC_db = error


    w = w - eta * dC_dw
    b = b - eta * dC_db


    loss = (1/(2*n)) * np.sum((w*x_train + b - y_train)**2)
    losses_sgd.append(loss)

print(f'Final w = {w:.4f}, b = {b:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(losses_sgd)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('SGD batch=1 (eta=0.01)')
plt.grid(True)
plt.show()

### Q5c - Divergence with eta = 0.5

Set eta = 0.5 with full-batch GD. Show the loss curve and explain using the gradient formulas from Q5a why it diverges.

> **Your answer:**

In [ ]:
w, b = 0.0, 0.0
eta = 0.5
losses_div = []

n = len(x_train)

for epoch in range(50):

    y_pred = w * x_train + b
    error = y_pred - y_train

    dC_dw = (1/n) * np.sum(error * x_train)
    dC_db = (1/n) * np.sum(error)

    w = w - eta * dC_dw
    b = b - eta * dC_db

    loss = (1/(2*n)) * np.sum((w*x_train + b - y_train)**2)
    losses_div.append(loss)

    if not np.isfinite(loss):
        print(f'Diverged at epoch {epoch}')
        break

plt.figure(figsize=(7, 4))
plt.plot(losses_div)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Full-Batch GD (eta=0.5) - Divergence')
plt.grid(True)
plt.show()

---
## Q6 - One-Hot to 4-Bit Binary Encoding Layer

A network classifies digits 0-9 with a 10-neuron one-hot output. Design a final **4-neuron layer** that converts this into a **4-bit binary** encoding (e.g. digit 5 outputs 0101). Specify the weight matrix and bias vector.

**Design and Reasoning:**

> **Your answer:**

In [ ]:
import numpy as np

W = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],  # 8's bit
    [0, 0, 0, 0, 1, 1, 1, 1, 0, 0],  # 4's bit
    [0, 0, 1, 1, 0, 0, 1, 1, 0, 0],  # 2's bit
    [0, 1, 0, 1, 0, 1, 0, 1, 0, 1]   # 1's bit
], dtype=float)

b_enc = np.zeros(4)
W     = None  # shape (4, 10)
b_enc = None  # shape (4,)

def threshold(x):
    return (x >= 0.5).astype(int)

print('Digit | Output | Expected | OK?')
print('-' * 35)
for digit in range(10):
    one_hot = np.zeros(10)
    one_hot[digit] = 1.0
    out = threshold(W @ one_hot + b_enc)
    expected = format(digit, '04b')
    ok = 'OK' if ''.join(map(str, out)) == expected else 'FAIL'
    print(f'  {digit}   |  {out}  |  {expected}  | {ok}')

---
## Q7 - Geometric Interpretation of 1D Gradient Descent

Consider C(v) as a function of a single scalar v.

1. Give a precise **geometric interpretation** of the update rule v = v - alpha * dC/dv.
2. How do the slope and the learning rate alpha physically dictate movement along the curve?

> **Your answer:**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

v_vals = np.linspace(-3, 5, 300)
C_v = (v_vals - 2)**2 + 1

# Gradient descent trajectory
alpha = 0.2
v = -2.0

trajectory_v = [v]
trajectory_C = [(v - 2)**2 + 1]

for _ in range(10):
    grad = 2 * (v - 2)      # dC/dv
    v = v - alpha * grad
    trajectory_v.append(v)
    trajectory_C.append((v - 2)**2 + 1)

plt.figure(figsize=(7, 4))
plt.plot(v_vals, C_v, label='C(v)')
plt.scatter(trajectory_v, trajectory_C, label='GD Steps')
plt.plot(trajectory_v, trajectory_C)
plt.xlabel('v')
plt.ylabel('C(v)')
plt.title('1D Gradient Descent')
plt.legend()
plt.grid(True)
plt.show()

---
## Q8 - Zero Hidden Layers: What Model Are You Really Training?

Suppose you remove all hidden layers, connecting 784 inputs directly to 10 outputs, trained with SGD.

1. What **mathematical model** have you effectively created?
2. Why is classification accuracy **fundamentally capped**?
3. What does this imply about the **necessity of hidden layers**?

> **Your answer:**If all hidden layers are removed and the 784 input pixels are connected directly to 10 output neurons, the network becomes a multiclass linear classifier (equivalently, multinomial logistic regression if softmax is used at the output). The output for each class is simply a weighted sum of the input pixels, so the decision boundaries between classes are linear hyperplanes in the 784-dimensional input space.

Classification accuracy is fundamentally capped because many real-world patterns, including handwritten digits, are not linearly separable. A linear model cannot learn complex features such as curves, loops, corners, or combinations of strokes; it can only form linear decision boundaries. As a result, there will always be some digits that cannot be correctly separated by any choice of weights.

This implies that hidden layers are necessary because they learn intermediate features and introduce nonlinearity. By composing multiple nonlinear transformations, hidden layers can create highly complex decision boundaries that separate classes which a purely linear model cannot, enabling much higher classification accuracy.
